# LOBO — Leave-One-Battery-Out (window-30 Mamba) — NCKH Table 2

Chạy `scripts/experiment_nowcast_lobo.py` trên 26 pin chuẩn của protocol:
mỗi fold giữ 1 pin ra test, train trên 25 pin còn lại (refit scaler mỗi fold — không leakage),
kết quả cuối = **MAE/RMSE mean ± std across folds** → điền Table 2 bài báo.

**Setup trước khi chạy:**
- Settings → Accelerator: **GPU P100** (hoặc T4) · Internet: **On**
- `+ Add Data` → dataset `nasa-battery-dataset` (có `cleaned_dataset/`)
- Add-ons → Secrets → `GITHUB_TOKEN` (nếu repo private)
- Chạy nền: **Save Version → Save & Run All (Commit)** — xong vào tab Output tải kết quả

Ước lượng: ~5–15 phút/fold trên P100 → 26 fold ≈ 3–6 giờ (vừa 1 session 9h).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU roi restart!"

## 2. Clone repo (branch `dev`)

In [ ]:
import subprocess

REPO_URL = "github.com/GSU26SE55/ai-module.git"
BRANCH = "dev"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@{REPO_URL}"
except Exception:
    print("Khong co secret GITHUB_TOKEN -> clone public")
    clone_url = f"https://{REPO_URL}"

subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                clone_url, "/kaggle/working/ai-module"], check=True)

# Xoa token khoi remote
subprocess.run(["git", "-C", "/kaggle/working/ai-module", "remote", "set-url",
                "origin", f"https://{REPO_URL}"], check=True)

# Ghi lai commit hash — so lieu bai bao phai trace duoc ve dung commit
commit = subprocess.run(["git", "-C", "/kaggle/working/ai-module", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Commit:", commit)

## 3. Dependencies + đường dẫn

In [ ]:
%pip install -q scipy scikit-learn

import os

REPO = "/kaggle/working/ai-module"
DATASET = "/kaggle/input/nasa-battery-dataset/cleaned_dataset"

if not os.path.isfile(f"{DATASET}/metadata.csv"):
    # Tim vi tri thuc te neu ten dataset/thu muc khac
    import glob
    hits = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
    assert hits, "Khong tim thay metadata.csv — kiem tra da Add Data dataset chua"
    DATASET = os.path.dirname(hits[0])

os.chdir(REPO)
print("DATASET =", DATASET)
print("metadata.csv:", os.path.isfile(f"{DATASET}/metadata.csv"))
print("CSV count:", len([f for f in os.listdir(f"{DATASET}/data") if f.endswith(".csv")]))

## 4. Smoke test — 1 fold, 2 epoch (~1–2 phút)

Xác nhận pipeline chạy được trước khi tốn giờ GPU. Phải thấy dòng
`[hold B0005] test N windows -> MAE ...` ở cuối.

In [ ]:
!python scripts/experiment_nowcast_lobo.py \
    --data-dir "{DATASET}" --epochs 2 --folds B0005

## 5. Full LOBO — 26 pin chuẩn của protocol

`--pool` giới hạn đúng 26 pin (loại B0036 nhiễu, B0049–52 corrupt, B0038–40 dự phòng
như protocol trong `scripts/preprocess.py`). Pin < 60 cycles sẽ tự bị skip — số fold
thực tế có thể ít hơn 26, log ghi rõ.

Seed 42 cố định trong script. Đây là cell lâu nhất (3–6 giờ).

In [ ]:
POOL = ("B0005,B0006,B0007,B0018,"
        "B0025,B0026,B0027,B0028,B0029,B0030,B0031,B0032,B0033,B0034,"
        "B0041,B0042,B0043,B0044,B0045,B0046,B0047,B0048,"
        "B0053,B0054,B0055,B0056")

!python scripts/experiment_nowcast_lobo.py \
    --data-dir "{DATASET}" --pool "{POOL}" --epochs 60

## 6. Parse log → bảng per-fold + summary (Table 2)

Đọc log mới nhất trong `logs/training/`, xuất CSV per-fold + dòng mean±std.

In [ ]:
import glob
import re

import pandas as pd

logs = sorted(glob.glob(f"{REPO}/logs/training/*.log"), key=os.path.getmtime)
assert logs, "Khong thay log — cell 5 da chay xong chua?"
log_path = logs[-1]
print("Log:", log_path)

pat = re.compile(r"\[hold (B\d+)\] test (\d+) windows -> MAE ([\d.]+)% \| RMSE ([\d.]+)%")
rows = []
with open(log_path) as f:
    for line in f:
        m = pat.search(line)
        if m:
            rows.append({"battery": m.group(1), "n_test_windows": int(m.group(2)),
                         "mae_pct": float(m.group(3)), "rmse_pct": float(m.group(4))})

# keep="last": neu smoke-fold B0005 (epochs=2) lot vao cung log thi giu ket qua full-run
df = pd.DataFrame(rows).drop_duplicates(subset="battery", keep="last").reset_index(drop=True)

print(df.to_string(index=False))
print("=" * 50)
print(f"LOBO over {len(df)} folds:")
print(f"  MAE  = {df.mae_pct.mean():.4f}% ± {df.mae_pct.std(ddof=0):.4f}%  "
      f"(min {df.mae_pct.min():.3f}, max {df.mae_pct.max():.3f})")
print(f"  RMSE = {df.rmse_pct.mean():.4f}% ± {df.rmse_pct.std(ddof=0):.4f}%")

os.makedirs("/kaggle/working/lobo_results", exist_ok=True)
df.to_csv("/kaggle/working/lobo_results/lobo_per_fold.csv", index=False)
with open("/kaggle/working/lobo_results/table2_summary.txt", "w") as f:
    f.write(f"commit: {commit}\nfolds: {len(df)}\n")
    f.write(f"MAE  = {df.mae_pct.mean():.4f}% +/- {df.mae_pct.std(ddof=0):.4f}%\n")
    f.write(f"RMSE = {df.rmse_pct.mean():.4f}% +/- {df.rmse_pct.std(ddof=0):.4f}%\n")

## 7. Gom log + kết quả để download (tab Output)

In [ ]:
import shutil

shutil.copytree(f"{REPO}/logs/training", "/kaggle/working/lobo_results/logs",
                dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/lobo_results", "zip", "/kaggle/working/lobo_results")
print("Created: /kaggle/working/lobo_results.zip — tai ve tu tab Output")

## Sau khi chạy xong

1. Tải `lobo_results.zip` từ tab **Output**
2. Giải nén vào `logs/nckh/lobo/` trong repo local, commit (số liệu bài báo phải reproducible)
3. `table2_summary.txt` = số điền thẳng vào **Table 2** · `lobo_per_fold.csv` = phụ lục per-battery

**Nếu session quá 9h:** tách cell 5 thành 2 run — giữ nguyên `--pool`, thêm
`--folds B0005,...,B0034` (run 1) và `--folds B0041,...,B0056` (run 2), rồi gộp 2 CSV.